In [28]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

In [29]:
df = pd.read_csv('cleaned_telco.csv')

 Select features relevant for prediction (exclude leakage columns like Churn Score, Churn Category/Reason, Customer Status)

In [30]:
features = ['Age','Tenure in Months','Monthly Charge','Total Charges','Satisfaction Score',
            'Number of Referrals','Contract','Internet Type','Payment Method',
            'Paperless Billing','Senior Citizen','Married','Dependents']

X = df[features].copy()
y = df['Churned']


Encode categoricals

In [31]:
cat_cols = ['Contract','Internet Type','Payment Method','Paperless Billing','Senior Citizen','Married','Dependents']
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:,1]

print("=== Classification Report ===")
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


=== Classification Report ===
              precision    recall  f1-score   support

           0       0.96      0.98      0.97       944
           1       0.96      0.91      0.93       374

    accuracy                           0.96      1318
   macro avg       0.96      0.95      0.95      1318
weighted avg       0.96      0.96      0.96      1318

ROC-AUC: 0.9926895450013595
Confusion Matrix:
 [[929  15]
 [ 34 340]]


Feature importance (coefficients)

In [33]:
coef_df = pd.DataFrame({'feature': X.columns, 'coefficient': model.coef_[0]})
coef_df['abs_coef'] = coef_df['coefficient'].abs()
coef_df = coef_df.sort_values('abs_coef', ascending=False)

print("=== Top 10 Features Driving Churn ===")
print(coef_df.head(10)[['feature','coefficient']])

coef_df.to_csv('feature_importance.csv', index=False)


=== Top 10 Features Driving Churn ===
                      feature  coefficient
4          Satisfaction Score    -6.456654
5         Number of Referrals    -1.619074
1            Tenure in Months    -1.241913
7           Contract_Two Year    -0.929839
14                Married_Yes     0.754017
15             Dependents_Yes    -0.685908
6           Contract_One Year    -0.468131
3               Total Charges     0.453628
9   Internet Type_Fiber Optic     0.340655
13         Senior Citizen_Yes     0.334730
